In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
spark = SparkSession.builder.appName("GTFS Silver").getOrCreate()

In [0]:
# Read the Parquet files from Silver layer
df_stop = spark.read.table("transport.silver.gtfs_stops")
df_trip = spark.read.table("transport.silver.gtfs_trips")
df_stop_times = spark.read.table("transport.silver.gtfs_stop_times")
df_route = spark.read.table("transport.silver.gtfs_routes")
df_calendar = spark.read.table("transport.silver.gtfs_calendar_dates")

df_stop_night = spark.read.table("transport.silver.gtfs_stops_night")
df_trip_night = spark.read.table("transport.silver.gtfs_trips_night")
df_stop_times_night = spark.read.table("transport.silver.gtfs_stop_times_night")
df_route_night = spark.read.table("transport.silver.gtfs_routes_night")

In [0]:
df_stop_night = (
    df_stop_night
    .withColumn(
        'stop_lon',
        F.regexp_replace('stop_lon', '<<<<<<<<<<<<<', '23.9833036')
        )
    )


In [0]:
df_calendar = df_calendar.withColumn("service_id", F.col("service_id").cast("string"))
df_trip = df_trip.withColumn("service_id", F.col("service_id").cast("string"))

df_total_jour = (
    df_stop_times.join(df_trip, "trip_id")
    .join(df_route, "route_id")
    .join(df_calendar, "service_id")
    .join(df_stop, "stop_id")
    .withColumn("jour_ou_nuit", F.lit('0'))
    .withColumn("service_id", F.col("service_id").cast("string"))
)

In [0]:
df_trip_night = df_trip_night.withColumn("route_id", F.col("route_id").cast("string"))
df_route_night = df_route_night.withColumn("route_id", F.col("route_id").cast("string"))
# stop_sequence : double côté night, integer côté jour
df_stop_times_night = df_stop_times_night.withColumn("stop_sequence", F.col("stop_sequence").cast("integer"))
df_stop_night = df_stop_night.withColumn("stop_lon", F.col("stop_lon").cast("double"))

df_total_nuit = (
    df_stop_times_night.join(df_trip_night, "trip_id")
    .join(df_route_night, "route_id")
    .join(df_stop_night, "stop_id")
    .withColumn("jour_ou_nuit", F.lit('1'))
    .withColumn("date", F.lit('20200101'))
    .withColumn('exception_type', F.lit(999))
    .withColumn("service_id", F.col("service_id").cast("string"))
)

# Df total

In [0]:
df_total = (
    df_total_jour
    .withColumn("date", F.col("date").cast("string"))
    .withColumn("date", F.to_date("date", "yyyyMMdd"))
)

In [0]:
df_total.display()

In [0]:
arrival_hour = F.substring("arrival_time", 1, 2).cast("int")
departure_hour = F.substring("departure_time", 1, 2).cast("int")

df_total = (
    df_total

    # Ajouter 1 jour si une des heures > 23
    .withColumn(
        "date",
        F.when(
            (arrival_hour > 23) | (departure_hour > 23),
            F.date_add(F.col("date"), 1)
        ).otherwise(F.col("date"))
    )

    # Corriger arrival_time
    .withColumn(
        "arrival_time",
        F.when(
            arrival_hour > 23,
            F.concat(
                F.lpad((arrival_hour - 24).cast("string"), 2, "0"),
                F.substring("arrival_time", 3, 6)
            )
        ).otherwise(F.col("arrival_time"))
    )

    # Corriger departure_time
    .withColumn(
        "departure_time",
        F.when(
            departure_hour > 23,
            F.concat(
                F.lpad((departure_hour - 24).cast("string"), 2, "0"),
                F.substring("departure_time", 3, 6)
            )
        ).otherwise(F.col("departure_time"))
    )
)


In [0]:
# supprimer la table si elle existe
spark.sql(f"DROP TABLE IF EXISTS transport.gold.gtfs_gold")

In [0]:
(
    df_total
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("transport.gold.gtfs_gold")
)

In [0]:
(
    df_total_nuit
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("transport.gold.gtfs_gold_nuit")
)